# The configuration registry

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

Everything Larvaworld can run is stored, not written : experiments, models, environments, lab data
formats, reference datasets and batch protocols all live in one **registry**, each entry under a
unique ID. When you write `ExpRun(experiment="chemorbit")`, that string is a registry lookup.

This notebook is about the registry itself : what kinds of configuration it holds, how an entry is
retrieved, and the full lifecycle of making one of your own - copy an existing entry, change it,
store it under a new ID, and delete it again.

**What you will be able to do afterwards**

- Name the configuration types (*conftypes*) Larvaworld manages and list the entries of each.
- Retrieve a stored configuration either as a plain nested dictionary or as a live object.
- Copy, modify and store a configuration under your own ID, and remove it again.
- Say where on disc the registry keeps each conftype.

**Prerequisites** : [The Python API](../1_getting_started/python_api_basics.ipynb).

**Cost** : seconds.

**Switches in this notebook**

| switch | default | what it turns on |
|---|---|---|
| `RUN_WRITE_DEMOS` | `False` | writing to and deleting from the on-disc configuration files |
| `RUN_VISUAL_DEMOS` | `False` | rendering every stored environment in a pygame window |

## Setup

Importing `larvaworld` initializes the registry : part of it is loaded from disc, the rest is built
on the fly the first time it is asked for.

In [1]:
%matplotlib inline

%load_ext param.ipython

import larvaworld
from larvaworld.lib import reg
from larvaworld.lib.reg.generators import EnvConf

larvaworld.VERBOSE = 1

# Tutorial safety switches
RUN_WRITE_DEMOS = False  # writes to the on-disc conf dict
RUN_VISUAL_DEMOS = False  # opens pygame windows

Welcome to the param IPython extension! (https://param.holoviz.org/)
Available magics: %params


Initializing larvaworld registry


Registry configured!


## Section 1 : Conftypes

A **conftype** is a category of configuration. Each one answers a different question about a run :

| conftype | what it configures |
|---|---|
| `Env` | the arena : geometry, food, borders, sensory landscapes |
| `Model` | a larva : body, physics and brain modules |
| `Exp` | a whole experiment : environment + larva groups + what to record |
| `Batch` | a parameter sweep over experiments |
| `Ga` | a genetic-algorithm optimization |
| `Ref` | reference datasets, by ID |
| `LabFormat` | how a particular lab's raw tracking files are laid out |
| `Trial` | the temporal protocol of an experiment - its epochs and their timing |

The list is available programmatically, which is the safest way to see what the installed version
supports.

In [2]:
print(larvaworld.CONFTYPES)

['Env', 'LabFormat', 'Ref', 'Model', 'Trial', 'Exp', 'Batch', 'Ga']


Each conftype is managed by one `ConfType` instance. They live in `reg.conf`, an `AttrDict`, so the
manager for a conftype is reachable both by attribute and by key - the two lines below are the same
object.

In [3]:
conftype = "Env"
ct = reg.conf.Env

assert ct is reg.conf[conftype]
assert ct.conftype == conftype

print(f"The manager for {conftype!r} is a {ct.__class__.__name__}")

The manager for 'Env' is a ConfType


## Section 2 : What a manager holds

A `ConfType` keeps its entries in a dictionary, `ct.dict`, persisted at `ct.path_to_dict`. The keys
of that dictionary are the configuration IDs, listed in `ct.confIDs`.

Knowing the path matters in practice : that file is what you copy when you want to move your
configurations to another machine, and what you delete when you want to go back to the defaults
that ship with the package.

In [4]:
print(f"Storage type : {ct.dict.__class__.__name__}")
print(f"Stored at    : {ct.path_to_dict}")

Storage type : AttrDict
Stored at    : C:\Users\Panos\larvaworld_GH_nawrotlab\larvaworld\src\larvaworld/lib/reg/confDicts/Env.txt


In [5]:
print("Stored configurations per conftype :")
print()
for k in larvaworld.CONFTYPES:
    ids = reg.conf[k].confIDs
    print(f"  {k:12s} {len(ids):4d}   e.g. {ids[:3]}")

Stored configurations per conftype :

  Env            35   e.g. ['4corners', 'CS_UCS_off_food', 'CS_UCS_on_food']
  LabFormat       5   e.g. ['Arguello', 'Berni', 'DeepLabCut']
  Ref            11   e.g. ['Chris.larvae_single', 'DeepLabCut.TopDown-2023-07-05', 'DeepLabCut.TopDown-2024-02-17']
  Model         612   e.g. ['CON_CON_DEF_BR', 'CON_CON_DEF_DEF', 'CON_CON_PHI_BR']
  Trial           3   e.g. ['default', 'odor_preference', 'odor_preference_short']
  Exp            58   e.g. ['4corners', 'MB_patch_grid', 'PItest_off']
  Batch          10   e.g. ['PItest_off', 'PItrain', 'PItrain_mini']
  Ga              5   e.g. ['chemorbit', 'exploration', 'interference']


## Section 3 : Retrieving a configuration

There are two ways to get an entry out, and the difference matters :

- **`ct.getID(id)`** returns the stored **nested dictionary** - what was written to disc. Use it
  when you want to read or modify values.
- **`ct.get(id)`** returns a live **configuration object** of the corresponding class, with typed
  parameters, bounds, defaults and documentation. Use it when you want validation, or the methods
  the class provides.

In [6]:
envID = ct.confIDs[1]

# as a nested dictionary
entry = ct.getID(envID)
print(f"{envID!r} as a {entry.__class__.__name__} :")
entry.print()

'CS_UCS_off_food' as a AttrDict :
     arena : 
          dims : (0.1, 0.1)
          geometry : circular
          torus : False
     border_list : 
     food_params : 
          food_grid : None
          source_groups : 
          source_units : 
               CS : 
                    amount : 0.0
                    can_be_carried : False
                    can_be_displaced : False
                    color : red
                    group : None
                    odor : 
                         id : CS
                         intensity : 2.0
                         spread : 0.01
                    pos : (-0.04000000000000001, 0.0)
                    radius : 0.003
                    regeneration : False
                    regeneration_pos : None
                    substrate : 
                         composition : 
                              glucose : 0.0
                              dextrose : 0.0
                              saccharose : 0.0
                   

In [7]:
# ... and as a configuration object
obj = ct.get(envID)
print(f"{envID!r} as an object : {obj.__class__.__name__}")
print("Parameters :", obj.param_keys())

'CS_UCS_off_food' as an object : EnvConf
Parameters : ['arena', 'border_list', 'food_params', 'odorscape', 'thermoscape', 'windscape']


The object form is what `%params` documents : every field with its type, default, bounds and
docstring. This is the reference you want open while building a configuration by hand.

In [8]:
%params EnvConf

Nested parts are objects too, so you can drill into one and document it on its own - here the odor
landscape of that environment.

In [9]:
obj2 = ct.get(ct.confIDs[2])
print(f"{ct.confIDs[2]!r} odorscape :", obj2.odorscape.__class__.__name__)
%params obj2.odorscape

'CS_UCS_on_food' odorscape : GaussianValueLayerUnit


## Section 4 : Making one of your own

The usual workflow is not to build a configuration from nothing, but to copy the nearest existing
one and change what your question needs. `get_copy()` gives you an independent copy, so editing it
cannot corrupt the stored entry.

In [10]:
new_conf = entry.get_copy()
new_conf.arena.dims = (0.5, 0.1)

print(f"Stored arena : {entry.arena.dims}")
print(f"New arena    : {new_conf.arena.dims}")

Stored arena : (0.1, 0.1)
New arena    : (0.5, 0.1)


Storing it is `setID`, deleting it is `delete`. Both write to the file printed in Section 2, which
is why they are behind a switch here : running this notebook should not change your registry unless
you ask it to.

In [11]:
if RUN_WRITE_DEMOS:
    new_id = "my_long_arena"

    assert new_id not in ct.confIDs
    ct.setID(id=new_id, conf=new_conf)
    assert new_id in ct.confIDs
    print(f"Stored {new_id!r}")

    ct.delete(id=new_id)
    assert new_id not in ct.confIDs
    print(f"Deleted {new_id!r}")
else:
    print("Set RUN_WRITE_DEMOS = True to store and delete a configuration.")

Set RUN_WRITE_DEMOS = True to store and delete a configuration.


From then on the new ID behaves exactly like a built-in one : `reg.conf.Env.get("my_long_arena")`,
or `env_params="my_long_arena"` inside an experiment configuration. This is also what the Portal's
*Stored Configurations* panels write to, so a configuration you build in the browser is immediately
available from Python, and the other way round.

## Section 5 : Seeing what a configuration describes

An `Env` configuration object can render itself, which is the quickest sanity check that the arena
you just built is the one you meant. It opens a pygame window, so it is off by default.

In [12]:
if RUN_VISUAL_DEMOS:
    for envID_i in ct.confIDs[:3]:
        ct.get(envID_i).visualize(duration=0.3)
else:
    print("Set RUN_VISUAL_DEMOS = True to render the stored environments.")

Set RUN_VISUAL_DEMOS = True to render the stored environments.


## Where to go next

- [Building an environment](environment_configuration.ipynb) - the `Env` conftype in detail.
- [Sensory landscapes](sensory_landscapes.ipynb) - the layers that go on top of an arena.
- [The Python API](../1_getting_started/python_api_basics.ipynb) - using a configuration in a run.
- Reference : [Experiment configuration pipeline](../../concepts/experiment_configuration_pipeline.md)
  explains how a stored entry becomes a running simulation.